# Laboratório 05: Dados Ausentes - Tratamento Univariado e Multivariado
**Disciplina:** Extração e Preparação de Dados (IBM8915)
**Objetivo:** Aprender a usar ferramentas de descarte (`dropna`), preenchimento simples (`fillna`) com estatísticas básicas (univariado) e algoritmos de machine learning via Scikit-Learn (`KNNImputer`) (multivariado) para tratamento assertivo de dados defeituosos.

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('ggplot')
import warnings
warnings.filterwarnings('ignore')

## Parte 1: Exemplo Guiado - A "Faca" e a "Seringa"
Neste exemplo rápido, usaremos o `.dropna()` (a faca) para amputar o que não tem salvação e o `.fillna()` (a seringa) para curar colunas numéricas e categóricas do nosso DataFrame de pacientes.

In [3]:
# Criando dados fictícios
df_exemplo = pd.DataFrame({
    'Paciente': ['Ana', 'Bruno', 'Carlos', 'Diana', 'Eduardo'],
    'Idade': [25, np.nan, 42, 38, np.nan],         # Numérica
    'Pressao': [120, 125, 130, np.nan, 140],       # Numérica
    'Tipo_Sanguineo': ['O+', 'A-', np.nan, 'O+', np.nan], # Categórica
    'Coluna_Perdida': [np.nan, np.nan, np.nan, np.nan, np.nan] # 100% Nula
})

print("Estado Original:\n", df_exemplo, "\n")

# 1. A Faca: Descartando colunas 100% vazias (axis=1 atinge colunas)
df_exemplo.dropna(axis=1, how='all', inplace=True)

# 2. A Seringa (Numéricos): Preenchendo Idade com a Mediana
mediana_idade = df_exemplo['Idade'].median()
df_exemplo['Idade'].fillna(mediana_idade, inplace=True)

# 3. A Seringa (Categóricos): Preenchendo Tipo Sanguíneo com a Moda
moda_sangue = df_exemplo['Tipo_Sanguineo'].mode()[0] #  pega a primeira moda se houver empate
df_exemplo['Tipo_Sanguineo'].fillna(moda_sangue, inplace=True)

print("Estado Tratado:\n", df_exemplo)

Estado Original:
   Paciente  Idade  Pressao Tipo_Sanguineo  Coluna_Perdida
0      Ana   25.0    120.0             O+             NaN
1    Bruno    NaN    125.0             A-             NaN
2   Carlos   42.0    130.0            NaN             NaN
3    Diana   38.0      NaN             O+             NaN
4  Eduardo    NaN    140.0            NaN             NaN 

Estado Tratado:
   Paciente  Idade  Pressao Tipo_Sanguineo
0      Ana   25.0    120.0             O+
1    Bruno    NaN    125.0             A-
2   Carlos   42.0    130.0            NaN
3    Diana   38.0      NaN             O+
4  Eduardo    NaN    140.0            NaN


## Parte 2: Exercício Prático (Mão na Massa)
Abaixo geramos um pequeno dataset de imóveis (`df_imoveis`). 
**Sua Tarefa:**
1. Descarte as colunas que possuam **mais de 70%** de valores nulos.
2. Preencha a coluna numérica `Preco` usando a estatística adequada (Média ou Mediana? Dica: Verifique se existem outliers absurdos com `.describe()`).
3. Preencha a coluna categórica `Bairro` com o valor mais comum (Moda).

In [4]:
# NÃO ALTERE ESTE CÓDIGO
np.random.seed(42)
df_imoveis = pd.DataFrame({
    'ID_Imovel': range(1, 101),
    'Bairro': np.random.choice(['Centro', 'Jardins', 'Vila Nova'], 100, p=[0.5, 0.3, 0.2]),
    'Preco': np.append(np.random.normal(300000, 50000, 95), [1500000, 2000000, 2500000, 3000000, 5000000]), # Com outliers milionários
    'Comissao_Corretor_Morta': [np.nan] * 100 # Coluna inútil
})

# Injetando nulos
df_imoveis.loc[np.random.choice(100, 15, replace=False), 'Preco'] = np.nan
df_imoveis.loc[np.random.choice(100, 20, replace=False), 'Bairro'] = np.nan

In [ ]:
# ESCREVA SEU CÓDIGO AQUI
# Passo 1: Use dropna para derrubar colunas com mais de 70% de nulos
limite_nulos = 0.70
df_imoveis.dropna(axis=1, thresh=int(len(df_imoveis) * (1 - limite_nulos)), inplace=True)

# Passo 2: Decida entre .mean() e .median() para 'Preco' e faça o fillna
# Com outliers milionários, usamos Mediana (mais robusta)
mediana_preco = df_imoveis['Preco'].median()
df_imoveis['Preco'].fillna(mediana_preco, inplace=True)

# Passo 3: Use .mode() para preencher os bairros ausentes
moda_bairro = df_imoveis['Bairro'].mode()[0]
df_imoveis['Bairro'].fillna(moda_bairro, inplace=True)

print("Nulos restantes:")
print(df_imoveis.isna().sum())
display(df_imoveis.head())

---
## Parte 3: Tratamento Multivariado (Avançado com Scikit-Learn)
Em vez de olhar apenas para a coluna com valores ausentes (univariado), a imputação multivariada utiliza as **outras variáveis** da mesma linha para prever o valor faltante. Vamos usar o `KNNImputer`, que acha os exemplos mais "parecidos" (vizinhos) para inferir a lacuna.

In [ ]:
from sklearn.impute import KNNImputer

# Criando um pequeno DataFrame de exemplo
df_knn = pd.DataFrame({
    'Salario': [5000, 5200, 9000, 5100, 8900],
    'Idade': [25, 27, 45, 26, np.nan],  # Alguém esqueceu a idade do último funcionário
    'Tempo_Empresa': [1, 2, 15, 1, 14]
})
print("Antes da Imputação (KNN):\n", df_knn, "\n")

# O KNNImputer substitui o valor ausente usando a média dos k vizinhos mais próximos
imputador_knn = KNNImputer(n_neighbors=2)
df_imputado_numpy = imputador_knn.fit_transform(df_knn)

# O Scikit-Learn retorna um array NumPy! Precisamos recriar o DataFrame
df_knn_tratado = pd.DataFrame(df_imputado_numpy, columns=df_knn.columns)
print("Após a Imputação (KNN):\n", df_knn_tratado)

### Exercício Prático: KNNImputer
Lembra da coluna `Preco` do nosso dataset `df_imoveis` (na Parte 2)? Se apenas preenchermos com a mediana global, ignoramos se a casa é uma mansão gigante ou um estúdio diminuto.
Vamos preencher o Preco agora de forma inteligente, usando a correlação com as variáveis numéricas fictícias adicionadas: `Metragem` e `Numero_Quartos`.

In [ ]:
# NÃO ALTERE ESTE CÓDIGO - Gerando os novos dados correlacionados
np.random.seed(99)
df_imoveis_knn = pd.DataFrame({
    'Metragem': np.random.randint(40, 200, 100),
    'Numero_Quartos': np.random.randint(1, 4, 100)
})
# O preço verdadeiro será uma função clara da metragem e dos quartos
df_imoveis_knn['Preco'] = (df_imoveis_knn['Metragem'] * 5000) + (df_imoveis_knn['Numero_Quartos'] * 15000) + np.random.normal(0, 10000, 100)

# Injetando buracos apenas no Preco (deletando 20% das informações de preço da base)
df_imoveis_knn.loc[np.random.choice(100, 20, replace=False), 'Preco'] = np.nan
print("Amostra dos Dados Esburacados (Veja que as linhas com Preço nulo possuem Metragem):")
display(df_imoveis_knn.head(10))

In [ ]:
# ESCREVA SEU CÓDIGO AQUI

# Passo 1: Instancie o KNNImputer (use n_neighbors=3 ou 5 para teste)
imputador_knn = KNNImputer(n_neighbors=5)

# Passo 2: Aplique o fit_transform no df_imoveis_knn e guarde o resultado (que será um array numpy)
resultado_numpy = imputador_knn.fit_transform(df_imoveis_knn)

# Passo 3: Recrie o Pandas DataFrame com os nomes originais das colunas
df_imoveis_knn_tratado = pd.DataFrame(resultado_numpy, columns=df_imoveis_knn.columns)

# Passo 4: Exiba o novo dataset tratado para validar se os preços foram preenchidos (use .head(10))
print("Nulos restantes:", df_imoveis_knn_tratado.isna().sum().sum())
display(df_imoveis_knn_tratado.head(10))

---
## Parte 4: Desafio para Casa - O Pipeline do Engenheiro de Dados
Este desafio consolida as **Aulas 02 a 07**. Rode a célula abaixo para gerar um arquivo CSV no seu disco chamado `ecommerce_messy.csv`.

**A Missão:**
1. **(Aula 02):** Carregue o arquivo lidando com o separador `;` e o encoding `latin-1`.
2. **(Aula 05):** Audite e remova as linhas inteiramente duplicadas.
3. **(Aula 04):** Converta a coluna `Categoria` para o tipo `category` para salvar memória.
4. **(Aula 06):** Plote um `sns.heatmap` para ver onde estão os nulos.
5. **(Aula 07):** Faça a imputação: `Idade` (Numérica) e `Categoria` (Texto).
6. **(Aula 04):** Calcule o Ticket Médio (`Renda`) agrupado por `Categoria`.

In [ ]:
# NÃO ALTERE ESTE CÓDIGO - Geração de dados sujos reais no disco
np.random.seed(10)
n = 800
df_desafio = pd.DataFrame({
    'ID_Cliente': np.random.randint(1000, 9999, n),
    'Idade': np.random.normal(35, 10, n),
    'Renda': np.random.normal(5000, 2000, n),
    'Categoria': np.random.choice(['Eletrônicos', 'Móveis', 'Vestuário'], n)
})

# Inserindo buracos (NaNs)
df_desafio.loc[np.random.choice(n, 120, replace=False), 'Idade'] = np.nan
df_desafio.loc[np.random.choice(n, 60, replace=False), 'Categoria'] = np.nan

# Inserindo linhas duplicadas
df_desafio = pd.concat([df_desafio, df_desafio.sample(45)], ignore_index=True)

# Exportando para o formato legado "Brasileiro"
df_desafio.to_csv('ecommerce_messy.csv', sep=';', encoding='latin-1', index=False)
print("Arquivo 'ecommerce_messy.csv' gerado na sua pasta local!")

In [ ]:
# ESCREVA SEU CÓDIGO AQUI

# 1. Carregue o CSV (read_csv com sep e encoding corretos)
df_eco = pd.read_csv('ecommerce_messy.csv', sep=';', encoding='latin-1')
print(f"Shape após carga: {df_eco.shape}")

# 2. Remova duplicatas (.drop_duplicates)
df_eco.drop_duplicates(inplace=True)
print(f"Shape após remover duplicatas: {df_eco.shape}")

# 3. Otimização de tipo (.astype('category'))
df_eco['Categoria'] = df_eco['Categoria'].astype('category')

# 4. Heatmap do Seaborn (sns.heatmap)
plt.figure(figsize=(8, 4))
sns.heatmap(df_eco.isna(), cbar=False, yticklabels=False, cmap='viridis')
plt.title("Mapa de Nulos")
plt.tight_layout()
plt.show()

# 5. Tratamento Univariado (fillna com mediana/moda)
df_eco['Idade'].fillna(df_eco['Idade'].median(), inplace=True)
df_eco['Categoria'].fillna(df_eco['Categoria'].mode()[0], inplace=True)
print(f"Nulos restantes:\n{df_eco.isna().sum()}")

# 6. Análise Agrupada (groupby)
ticket_medio = df_eco.groupby('Categoria')['Renda'].mean().sort_values(ascending=False)
print("\nTicket Médio por Categoria:")
display(ticket_medio)

---
## Parte 5: Desafio Final - Sobreviventes do Titanic (KNNImputer)
Para fixar o conhecimento de imputação multivariada vamos usar o famoso dataset do Titanic, embutido na biblioteca Seaborn.
O problema crônico desse dataset é a coluna `age` (idade), que possui dezenas de valores nulos. Usar apenas a média geral distorce a realidade: passageiros da 1ª classe tendiam a ser mais velhos que os da 3ª classe, e pessoas que viajavam sozinhas tinham idades diferentes das que viajavam em grandes famílias.

**Sua Missão:**
1. Isolar um sub-dataset apenas com características numéricas relevantes: `survived`, `pclass`, `age`, `sibsp` (Irmãos/Cônjuges), `parch` (Pais/Filhos) e `fare` (Tarifa).
2. Instanciar o `KNNImputer` e treinar neste sub-dataset para preencher as idades.
3. Recriar o DataFrame e confirmar que não existem mais nulos na variável `age` usando `.info()` ou `.isna().sum()`.

In [ ]:
# Carregando os dados originais do Titanic
df_titanic = sns.load_dataset('titanic')

print("Quantidade de Nulos Originais:")
display(df_titanic.isna().sum())

print("\nAmostra dos dados (note as idades NaN):")
display(df_titanic[df_titanic['age'].isna()].head())

In [ ]:
# ESCREVA SEU CÓDIGO AQUI

# Passo 1: Crie df_titanic_num selecionando apenas as colunas relevantes
df_titanic_num = df_titanic[['survived', 'pclass', 'age', 'sibsp', 'parch', 'fare']].copy()

# Passo 2: Instancie o KNNImputer (ex: 5 vizinhos)
imputer = KNNImputer(n_neighbors=5)

# Passo 3: Aplique o imputer.fit_transform() no df_titanic_num
resultado_numpy = imputer.fit_transform(df_titanic_num)

# Passo 4: Converta a matriz numpy de volta para um DataFrame do Pandas
df_titanic_tratado = pd.DataFrame(resultado_numpy, columns=df_titanic_num.columns)

# Passo 5: Exiba a quantidade de nulos na nova base (.isna().sum()) para validar o sucesso!
print("Nulos após KNNImputer:")
display(df_titanic_tratado.isna().sum())